## Evaluation pipeline for the microlane experiment

In [1]:
# First consider all the variables
# The input image gets resized to a particular level
# Then create a pipeline to feed data into the model
# AFter this process is completed, then process the data
# Then after the processing is done find a way to take output from the model
# Then, convert the output to relevant format, and store it for future use
# Apply relevant computations

In [2]:
# Imports of the Core Packages
import json, sys, time, pytz
import os, yaml,random 
import numpy as np
from datetime import datetime
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [3]:
# Import custom libraries located at different folder location + configs
from microlane.utils.metrics import *
from microlane.datasets.tusimple import TuSimple
from microlane.models.lanenet2.model import LaneNet2
from microlane.schema.output import ModelPrediction
from microlane.schema.sample import Sample
from microlane.utils.load_image import load_image_from_sample
from microlane.utils.experiment import ExperimentEvaluate

In [4]:
# First Load the Configuation file
with open("configs/config.yaml", "r") as file:
    config = yaml.safe_load(file)

### Pre Processing Part

In [5]:
# First initialise the dataset
# Then load the dataset
dataset = TuSimple(
    
        folder_path=config['data']['datasets']['tusimple']['path'],
        
        annotation_file_path=config['data']['datasets']['tusimple']['annotation_file']
    )

data = dataset.load(number=100)

In [6]:
# So, basically now we will import the model
# model = LaneNet2() type and what we will do is, run 
# Run model.inference(formatted_dataset)
# Ensure that Docker Engine Is Running in background

model = LaneNet2(
    
    container_folder=config['models']['lanenet2']['container_folder'],
    
    image_name=config['models']['lanenet2']['image_name']
    
)

Initializing container on port  8000
/home/suyog/desktop/projects/microlane/microlane/models/lanenet2/lanenet2
Image 'lanenet2_image:latest' already exists, skipping build.
Container already running: 15b340544e65


In [7]:
experiment = ExperimentEvaluate(
    
    experiment_name="initial Testing"
    
)

### Models and Datasets Loaded, Now Processing Part

In [70]:
# Print some basic information of our data

print(f"Total items: {len(data)}\n")

random_image_index = random.randint(0, len(data)-1)

item = data[random_image_index]
print(f"Index        : {random_image_index}")
print(f"Image Path   : {item.image_path}")
print(f"h_samples    : {item.h_samples}")
print(f"lanes        : {item.lanes}")

Total items: 100

Index        : 67
Image Path   : /home/suyog/assets/datasets/TuSimple/TUSimple/test_set/clips/0530/1492626682834154260_0/20.jpg
h_samples    : [160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260, 270, 280, 290, 300, 310, 320, 330, 340, 350, 360, 370, 380, 390, 400, 410, 420, 430, 440, 450, 460, 470, 480, 490, 500, 510, 520, 530, 540, 550, 560, 570, 580, 590, 600, 610, 620, 630, 640, 650, 660, 670, 680, 690, 700, 710]
lanes        : [[-2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, 769, 713, 680, 648, 628, 608, 589, 569, 550, 532, 514, 496, 478, 460, 442, 426, 412, 397, 383, 369, 355, 341, 327, 312, 298, 284, 270, 256, 242, 227, 213, 199, 185, 171, 157, 142, 128, 114, 100, 86, 72, 57, 43, 29, 15], [-2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, 850, 816, 800, 796, 794, 795, 800, 806, 812, 817, 823, 831, 839, 847, 855, 863, 871, 879, 887, 895, 903, 911, 919, 927, 935, 943, 951, 959, 967, 975, 983, 991, 999, 1007, 1015, 1023, 1031, 1039, 1047, 1055, 1063, 1071, 1079, 1087, 1

In [71]:

loaded_image = load_image_from_sample(item)

In [72]:
# We are basically sending a loaded sample with actual image tensor in the memory

response = model.predict(loaded_image)

In [73]:
prediction = response.json()

ModelOutput = ModelPrediction (
    
    sample=Sample(
        
            image=prediction['sample']['image'],
            image_path=prediction['sample']['image_path'],
            h_samples=prediction['sample']['h_samples'],
            lanes=prediction['sample']['lanes'],
            blur=prediction['sample']['blur'],
            brightness=prediction['sample']['brightness'],
            zoom=prediction['sample']['zoom'],
            rotation=prediction['sample']['rotation']

        ),
    
    lanes=prediction['lanes'],
        
    inference_time=prediction['inference_time']
    
)

In [74]:

experiment.store_prediction(ModelOutput)

In [75]:
experiment.visualize_prediction(ModelOutput)

'results/2026_04_16__09_29_15_initial_testing/visualization_0010.png'